In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 7
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)
    
    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.428221488335385

Trial 1 =========================================
13.417488316511063

Trial 2 =========================================
18.632359139419577



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


Trial 3 =========================================
14.288738291051644

Trial 4 =========================================
14.442158808128578

Trial 5 =========================================
13.224878875322974

Trial 6 =========================================
15.3869820523045

Trial 7 =========================================
15.929222010399386

Trial 8 =========================================
13.41972795001471

Trial 9 =========================================
14.43868501215935

Trial 10 =========================================
13.195106358701295

Trial 11 =========================================
17.57138671163402

Trial 12 =========================================
13.913554051207136

Trial 13 =========================================
16.241314203081213

Trial 14 =========================================
14.156539011646062

Trial 15 =========================================
16.02956264327333

Trial 16 =========================================
14.626945967896903



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 17 =========================================
16.12080270377876

Trial 18 =========================================
14.848469712295499

Trial 19 =========================================
14.300556641425523

Trial 20 =========================================
13.885726131813387

Trial 21 =========================================
15.39413854612002

Trial 22 =========================================
16.232118075723086

Trial 23 =========================================
13.909538108664417

Trial 24 =========================================
13.012973922665504

Trial 25 =========================================
15.387286838209821

Trial 26 =========================================
17.52029299789203



...

Trial 62 =========================================
15.929222010399386

Trial 63 =========================================
13.378594545768655

Trial 64 =========================================
18.733721856650714

Trial 65 =========================================
14.30537253300239

Trial 66 =========================================
18.811448710948973

Trial 67 =========================================
17.535700154860937

Trial 68 =========================================
15.529338013528818

Trial 69 =========================================
13.392785775557208

Trial 70 =========================================
14.315432252315322

Trial 71 =========================================
15.334603591446488

Trial 72 =========================================
15.400620114657084

Trial 73 =========================================
16.23623602942975

Trial 74 =========================================
18.811524593571654

Trial 75 =========================================
15.39315211441393

Trial 76 

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 81 =========================================
16.148291480842214

Trial 82 =========================================
16.15512997426859

Trial 83 =========================================
17.467605817778587

Trial 84 =========================================
14.289089121274605



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 85 =========================================
16.157527371737896

Trial 86 =========================================
16.177269130655283

Trial 87 =========================================
15.395145443645232

Trial 88 =========================================
13.38675712152749

Trial 89 =========================================
15.370438264949685

Trial 90 =========================================
15.373875392480748

Trial 91 =========================================
16.283368267355186

Trial 92 =========================================
15.38901954988684

Trial 93 =========================================
14.636966966462108

Trial 94 =========================================
17.349354963645236

Trial 95 =========================================
15.357789197172245

Trial 96 =========================================
15.413396673800792

Trial 97 =========================================
16.484387143073782

Trial 98 =========================================
14.80829990938245

Trial 99 

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.813805528448157
Avg = 15.472175758752353
Std = 1.4293522024433662


In [6]:
print(y_max_arr.tolist())

[14.428221488335385, 13.417488316511063, 18.632359139419577, 14.288738291051644, 14.442158808128578, 13.224878875322974, 15.3869820523045, 15.929222010399386, 13.41972795001471, 14.43868501215935, 13.195106358701295, 17.57138671163402, 13.913554051207136, 16.241314203081213, 14.156539011646062, 16.02956264327333, 14.626945967896903, 16.12080270377876, 14.848469712295499, 14.300556641425523, 13.885726131813387, 15.39413854612002, 16.232118075723086, 13.909538108664417, 13.012973922665504, 15.387286838209821, 17.52029299789203, 15.929222010399386, 18.813805528448157, 15.929222010399386, 13.804935414491508, 16.13483792219072, 15.981040319951841, 13.900265689722778, 15.388284402259455, 17.55277970428349, 16.217417828061016, 13.887773916398103, 14.263547504009082, 15.929222010399386, 15.957515834010167, 18.7845658095689, 15.396892212553098, 15.38266350897706, 16.054465066394563, 14.433828622420313, 14.148245500999495, 18.810670829839, 16.284013130765793, 13.441854041503436, 14.2891798863727

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-7.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)